# Radiation bias correction for low-cost temperature sensors

Low-cost temperature devices (LCD) are increasingly used to densify urban meteorological networks at low deployment cost {cite}`muller2013sensors,hamidi2020state,wong2025government`. However, these devices often lack proper radiation shields, which causes their temperature readings to be systematically biased upward during daytime {cite}`bell2015good,chapman2016can,buchau2018modelling,cornes2020correcting,bosch2026revisiting`.

The `meteora.bias_correction` module provides tools to apply pre-trained correction models that remove this radiation-driven bias. Each model captures the radiation-to-temperature-bias relationship for a specific sensor type and can be shared as a [scikit-learn pipeline](https://scikit-learn.org/stable/modules/compose.html#pipeline) serialized with [skops](https://skops.readthedocs.io) on [Hugging Face Hub](https://huggingface.co/models).

This notebook shows the following pipeline:

1. Retrieve LCD temperature data from the AWEL network of Decentlab sensors in Zurich {cite}`awel2026lufttemperatur` using `meteora.clients.AWELClient`.
2. Retrieve reference automated weather station (AWS) air temperature and shortwave radiation data from the [MeteoSwiss automated observation network (SwissMetNet)](https://www.meteoswiss.admin.ch/weather/measurement-systems/land-based-stations/automatic-measurement-network.html) using `meteora.clients.MeteoSwissClient`.
3. Load a pre-trained correction model from Hugging Face Hub at [`martibosch/decentlab-bias-correction`](https://huggingface.co/martibosch/decentlab-bias-correction).
4. Apply the correction and compare the results.

The pre-trained model used here is fitted based on the temperature readings of a Decentlab sensor and the temperature and shortwave radiation measurements from [a reference AWS from SwissMetNet at Zollikofen (Switzerland)](https://www.meteoswiss.admin.ch/services-and-publications/applications/measurement-values.html#param=messwerte-lufttemperatur-10min&table=false&station=BER), collocated alongside further LCD models in an intercomparison field study in summer 2025 {cite}`bosch2026revisiting`.

```{note}
This notebook requires the `sk` and `xvec` optional extras as well as the `interpret` library for the pre-trained correction models. The requirements can be installed as in, e.g.:

pip install interpret meteora[sk,xvec]
```

In [ ]:
import contextily as cx
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from huggingface_hub import hf_hub_download
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from skops import io as skops_io

from meteora import clients, settings, utils
from meteora.bias_correction import (
    BestScaleRadiationTransformer,
    apply_bias_correction,
    load_correction_model,
    parse_hf_path,
)

figwidth, figheight = plt.rcParams["figure.figsize"]

In [ ]:
region = "Zurich, Switzerland"
start = "2023-07-01"
end = "2023-07-31"

## LCD sensor data

We can start by getting LCD data for our study area, in this case, the city of Zurich (Switzerland). The AWEL (Amt für Abfall, Wasser, Energie und Luft) network operates Decentlab LoRa temperature sensors across the Zurich agglomeration. We use the `AWELClient` to retrieve temperature measurements for July 2023:

In [ ]:
awel_client = clients.AWELClient(region)
lcd_ts_df = utils.long_to_wide(
    awel_client.get_ts_df(settings.ECV_TEMPERATURE, start=start, end=end)
)
lcd_stations_gdf = awel_client.stations_gdf
lcd_ts_df.head()

## Reference weather station data

The bias correction relies on co-located shortwave radiation measurements to predict the radiation-induced temperature offset. We use the `MeteoSwissClient` to fetch air temperature and global shortwave radiation from official MeteoSwiss automated weather stations in the same region:

In [ ]:
aws_client = clients.MeteoSwissClient(region)
aws_ts_df = aws_client.get_ts_df(
    [settings.ECV_TEMPERATURE, settings.ECV_RADIATION_SHORTWAVE],
    start=start,
    end=end,
)
aws_ts_df.head()

We can visualise both station networks on a shared map:

In [ ]:
colors = sns.color_palette()
fig, ax = plt.subplots()
aws_client.stations_gdf.to_crs(lcd_stations_gdf.crs).plot(
    ax=ax, color=colors[0], label="AWS (MeteoSwiss)"
)
lcd_stations_gdf.plot(ax=ax, color=colors[1], label="LCD (AWEL)")
ax.legend()
cx.add_basemap(ax, crs=lcd_stations_gdf.crs, attribution="")

*(C) OpenStreetMap contributors, Tiles style by Humanitarian OpenStreetMap Team hosted by OpenStreetMap France*

For the spatial matching between LCD stations and their nearest AWS reference (needed to assign each LCD sensor the correct radiation time series), we can convert the long-form data frame and station geo-data frame to a [vector data cube](data-structures.ipynb#vector-data-cubes):

In [ ]:
aws_ts_cube = utils.long_to_cube(aws_ts_df, aws_client.stations_gdf)
aws_ts_cube

## Applying bias correction

We call `apply_bias_correction`, passing the HuggingFace Hub repository string directly as the model. Since the pipeline is serialized with [skops](https://skops.readthedocs.io), it is recommended to first inspect the non-sklearn types it contains as a security step before deserialization:

In [ ]:
model_str = "martibosch/lcd-bias-correction/decentlab.skops"
repo_id, filename = parse_hf_path(model_str)

trusted = skops_io.get_untrusted_types(file=hf_hub_download(repo_id, filename))
print("Types to review before trusting:")
for t in trusted:
    print(f"  {t}")

After reviewing the listed types, we can pass the HuggingFace Hub repository alongside the LCD and AWS data to `apply_bias_correction`, which will download and deserialize the model automatically. For each LCD station (`lcd_stations_gdf`), the function will find the nearest AWS reference station (`aws_ts_cube`), extract its radiation time series, run it through the model pipeline to predict the radiation-induced temperature offset, and finally subtract that offset from the raw LCD readings:

In [ ]:
cor_ts_df = apply_bias_correction(
    lcd_ts_df,
    aws_ts_cube,
    model_str,
    lcd_stations_gdf=lcd_stations_gdf,
    trusted=trusted,
)
cor_ts_df.head()

Note that `ref_ts` can be any meteora data structure in wide (single reference station, flat time series index) or long (multiple reference station multi-indexed by station and time — in that order) form. Wide inputs apply the same correction to every LCD station (based on the radiation readings from the single reference station), whereas long inputs require `lcd_stations_gdf` and `ref_stations_gdf` so that each LCD station is matched to its nearest reference via a spatial join. When `ref_ts` is a vector data cube the geometry is used for the spatial join directly, so `ref_stations_gdf` is not needed.

## Results

We can now inspect the correction for a LCD station over a three day window in mid-July alongside the shortwave radiation of the nearest MeteoSwiss reference, to assess the differences between AWS and LCD data (raw and corrected) as well as how these relate to the incoming shortwave radiation:

In [ ]:
aws_wide = utils.long_to_wide(aws_ts_df)
station_id = lcd_ts_df.columns[0]
plot_slice = slice("2023-07-10", "2023-07-12")

rad_wide = aws_wide[settings.ECV_RADIATION_SHORTWAVE]
aws_temp_wide = aws_wide[settings.ECV_TEMPERATURE]
aws_station_id = rad_wide.columns[0]  # nearest AWS station (approximate)

fig, axes = plt.subplots(nrows=2, sharex=True, figsize=(2 * figwidth, figheight))

axes[0].plot(
    aws_temp_wide.loc[plot_slice, aws_station_id],
    label=f"AWS ({aws_station_id})",
    color="steelblue",
    alpha=0.8,
)
axes[0].plot(
    lcd_ts_df.loc[plot_slice, station_id],
    label="LCD (uncorrected)",
    color="tomato",
    alpha=0.8,
)
axes[0].plot(
    cor_ts_df.loc[plot_slice, station_id],
    label="LCD (corrected)",
    color="seagreen",
    linewidth=1.5,
    linestyle="--",
)
axes[0].set_ylabel("Temperature (\u00b0C)")
axes[0].legend()

axes[1].plot(
    rad_wide.loc[plot_slice, aws_station_id],
    color="goldenrod",
    label=f"Shortwave radiation ({aws_station_id})",
)
axes[1].set_ylabel("Radiation (W m\u207b\u00b2)")
axes[1].set_xlabel("Time")
axes[1].legend()

fig.tight_layout()

## Training a correction model

Besides supporting the application of pre-trained models (i.e., via `apply_bias_correction`), meteora also supports training custom bias correction models given a paired time series of shortwave radiation from a co-located reference AWS (`X_train`) and the observed temperature bias at the LCD sensor (`y_train`, i.e. LCD minus reference temperature). For the most part, this is essentially done using [scikit-learn's pipelines](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html), which is a sequence of data transformations with a final predictor model. We can actually inspect the structure of the pre-trained pipeline by loading it explicitly:

In [ ]:
model = load_correction_model(repo_id, filename, trusted=trusted)
model

We can see that the pipeline is composed of the `BestScaleRadiationTransformer` followed by an `ExplainableBosstingRegressor` {cite}`lou2013accurate,nori2019interpretml`. The `BestScaleRadiationTransformer` from meteora is based on the observation that the temperature bias of an unshielded sensor is driven not only by instantaneous shortwave radiation but also by its accumulation over preceding time (due to the thermal inertia of the sensor) {cite}`bell2015good,cornes2020correcting,beele2022quality`. Accordingly, the transformer evaluates a set of candidate rolling-sum windows (in minutes) and selects the one most correlated with the observed temperature bias during `fit`; `transform` then applies that window to a new radiation time series and returns the result as a single-column DataFrame ready for any scikit-learn-compatible regressor.

Below we illustrate the workflow with the MeteoSwiss radiation data and a synthetic bias of 3 mK per W m⁻²:

In [ ]:
rad_ser = aws_wide[settings.ECV_RADIATION_SHORTWAVE].iloc[:, 0].dropna()

# X_train: time + radiation columns; y_train: temperature bias (LCD - reference)
# in practice y_train comes from paired LCD–reference measurements
X_train = pd.DataFrame(
    {
        settings.TIME_COL: rad_ser.index,
        settings.ECV_RADIATION_SHORTWAVE: rad_ser.values,
    }
)
y_train = (
    0.003 * rad_ser.values
)  # synthetic: 3 mK per W m\u207b\u00b2 (for illustration)

window_minutes = [30, 60, 120, 240]
pipeline = Pipeline(
    [
        ("transformer", BestScaleRadiationTransformer(window_minutes)),
        ("regressor", LinearRegression()),
    ]
)
pipeline.fit(X_train, y_train)
print(f"Best radiation window: {pipeline['transformer'].best_scale_} min")

While `LinearRegression` works well as a first-order approximation, non-linear models may better capture the relationship between accumulated radiation and sensor bias {cite}`cornes2020correcting,beele2022quality,bosch2026revisiting`.

```python
skops_io.dump(pipeline, "my-device.skops")

from huggingface_hub import HfApi

api = HfApi()
api.create_repo("username/lcd-bias-correction", repo_type="model", exist_ok=True)
api.upload_file(
    path_or_fileobj="my-device.skops",
    path_in_repo="my-device.skops",
    repo_id="username/lcd-bias-correction",
)
```

## References

```{bibliography}
:filter: docname in docnames
```